# Kaggle E2E RAG evaluation

## 1. Purpose and validation boundary

Copy this notebook by itself to Kaggle. It clones the production repository before importing project code, installs only runtime dependencies, reads attached datasets from `/kaggle/input`, and writes runs/exports to `/kaggle/working`.

The committed default is **inspect** and performs no retrieval or model call. Local validation cannot prove Kaggle networking, GPU behavior, provider access, quota, FAISS compatibility, or benchmark correctness.

## 2. Editable Kaggle configuration

Update `REPO_URL`/`REPO_BRANCH`, attach the benchmark and retrieval datasets, set source paths when auto-discovery is ambiguous, and add provider values through Kaggle Secrets. Public Git is the primary workflow.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/PhuongThao-2005/TextMining.git"
REPO_BRANCH = "my"
REPO_DIR = Path("/kaggle/working/TextMining")

FORCE_RECLONE = False
PULL_IF_EXISTS = True
INSTALL_DEPENDENCIES = True

PRIVATE_REPOSITORY = False
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"

CONFIG_NAME = "LLM-BaseReasoning"

RUN_MODE = "smoke"  # "inspect", "smoke", or "full"
SMOKE_LIMIT = 5
EXISTING_RUN_DIR = None

BENCHMARK_SOURCE = "/kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark/qa_final.jsonl"
CORPUS_SOURCE = "/kaggle/input/datasets/nguyenlethienlyy/text-mining-data-preprocessed/documents.jsonl"
FAISS_INDEX_SOURCE = "/kaggle/input/datasets/kittrntunk/faiss-chunk-meta/index.faiss"
FAISS_PAYLOADS_SOURCE = "/kaggle/input/datasets/kittrntunk/faiss-chunk-meta/payloads.jsonl"
FAISS_MANIFEST_SOURCE = None
BM25_SHARDS_SOURCE = "/kaggle/input/bm25-tokenized/bm25"
BM25_EXPECTED_SHARDS = 10
GRAPH_SOURCE = "/kaggle/input/datasets/myvnthdim/kg-pkl/knowledge_graph.gpickle"

RUNS_ROOT = Path("/kaggle/working/evaluation_runs/ablation")
EXPORT_ROOT = Path("/kaggle/working/e2e_rag_outputs")

USE_GPU_WHEN_AVAILABLE = True
CREATE_EXPORT_ZIP = True

## 3. Kaggle filesystem and hardware detection

In [ ]:
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_INPUT_ROOT.is_dir() and KAGGLE_WORKING_ROOT.is_dir()
print({"kaggle_environment": IS_KAGGLE, "python": platform.python_version()})

## 4. Clone or update repository

Private access lazily reads one Kaggle Secret. The token is passed only as an in-memory HTTP header; it is never printed, written, or stored in the remote URL.

In [ ]:
REQUIRED_CHECKOUT_PATHS = (
    Path("configs/ablation_configs.yaml"),
    Path("scripts/run_ablation_config.py"),
    Path("src"),
)

def sanitize_git_error(value, sensitive_values=()):
    text = str(value)
    for secret in sensitive_values:
        if secret:
            text = text.replace(secret, "***")
    text = re.sub(r"https?://[^/@\\s]+@github\\.com", "https://github.com", text)
    text = re.sub(r"(?i)(authorization|token|secret)(\\s*[:=]\\s*)([^\\s,;]+)", r"\\1\\2***", text)
    return text[-1000:]

def _git(args, *, cwd=None, sensitive_values=()):
    completed = subprocess.run(
        ["git", *[str(arg) for arg in args]],
        cwd=str(cwd) if cwd else None,
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode:
        detail = sanitize_git_error(completed.stderr or completed.stdout or "Git command failed.", sensitive_values)
        raise RuntimeError(f"Git operation failed safely: {detail}")
    return completed.stdout.strip()

def valid_checkout(repo_dir):
    root = Path(repo_dir)
    return (root / ".git").is_dir() and all((root / relative).exists() for relative in REQUIRED_CHECKOUT_PATHS)

def _private_token(secret_name):
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError as exc:
        raise RuntimeError("Private repository access requires Kaggle Secrets.") from exc
    try:
        return UserSecretsClient().get_secret(secret_name)
    except Exception:
        raise RuntimeError(f"Private repository secret {secret_name!r} is unavailable.") from None

def _safe_remove_checkout(repo_dir):
    target = Path(repo_dir).resolve()
    forbidden = {Path("/").resolve(), Path("/kaggle").resolve(), Path("/kaggle/working").resolve()}
    if target in forbidden or not target.name:
        raise RuntimeError("Refusing to remove a broad Kaggle directory.")
    if target.exists():
        shutil.rmtree(target)

def ensure_repository(
    repo_url,
    branch,
    repo_dir,
    *,
    force_reclone=False,
    pull_if_exists=True,
    private_repository=False,
    token_loader=None,
):
    target = Path(repo_dir)
    if force_reclone:
        _safe_remove_checkout(target)

    token = None
    if private_repository:
        token = (token_loader or (lambda: _private_token(GITHUB_TOKEN_SECRET_NAME)))()
        if not token:
            raise RuntimeError("Private repository access was requested but its secret is missing.")
    auth_args = ["-c", f"http.extraHeader=Authorization: Bearer {token}"] if token else []
    sensitive = (token,) if token else ()

    if target.exists():
        if not valid_checkout(target):
            raise RuntimeError(f"Configured repository directory is not a valid checkout: {target.name}")
        if pull_if_exists:
            _git(["checkout", branch], cwd=target, sensitive_values=sensitive)
            _git([*auth_args, "pull", "--ff-only", "origin", branch], cwd=target, sensitive_values=sensitive)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        try:
            _git([*auth_args, "clone", "--branch", branch, "--single-branch", repo_url, str(target)], sensitive_values=sensitive)
        finally:
            if (target / ".git").is_dir():
                _git(["remote", "set-url", "origin", repo_url], cwd=target, sensitive_values=sensitive)

    if not valid_checkout(target):
        raise RuntimeError("Repository checkout is missing required production files.")
    _git(["remote", "set-url", "origin", repo_url], cwd=target, sensitive_values=sensitive)
    commit = _git(["rev-parse", "HEAD"], cwd=target, sensitive_values=sensitive)
    return {"repository_path": str(target.resolve()), "branch": branch, "git_commit": commit}

repository_info = ensure_repository(
    REPO_URL,
    REPO_BRANCH,
    REPO_DIR,
    force_reclone=FORCE_RECLONE,
    pull_if_exists=PULL_IF_EXISTS,
    private_repository=PRIVATE_REPOSITORY,
)
print(repository_info)

## 5. Change working directory and configure imports

In [ ]:
os.chdir(REPO_DIR)
for import_root in (REPO_DIR, REPO_DIR / "src"):
    value = str(import_root)
    if value not in sys.path:
        sys.path.insert(0, value)
print({"working_directory": str(Path.cwd()), "import_root": str(REPO_DIR)})

## 6. Install and validate dependencies

The repository currently has no complete package/build metadata, so this cell installs the smallest selected-runtime set. It does not install pandas, pyarrow, Streamlit, CUDA, or PyTorch explicitly.

In [ ]:
def _module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        return False

def setup_dependencies(install=True, run_mode="inspect"):
    requirements = [("yaml", "PyYAML")]
    if run_mode in {"smoke", "full"}:
        requirements.extend([
            ("openai", "openai"),
            ("faiss", "faiss-cpu"),
            ("sentence_transformers", "sentence-transformers"),
            ("rank_bm25", "rank-bm25"),
        ])
    missing = [package for module, package in requirements if not _module_available(module)]
    if install and missing:
        completed = subprocess.run(
            [sys.executable, "-m", "pip", "install", *missing],
            check=False,
            capture_output=True,
            text=True,
        )
        if completed.returncode:
            detail = (completed.stderr or completed.stdout or "pip failed")[-1200:]
            raise RuntimeError(f"Dependency installation failed: {detail}")
    unresolved = [module for module, _ in requirements if not _module_available(module)]
    if unresolved:
        print("Restart the Kaggle session and rerun from the first cell.")
    return {"required": [module for module, _ in requirements], "missing_after_setup": unresolved}

dependency_status = setup_dependencies(INSTALL_DEPENDENCIES, RUN_MODE)
print(dependency_status)

## 7. Load Kaggle Secrets

Existing environment values win. Diagnostics expose only `configured` or `missing`.

In [ ]:
SECRET_MAPPING = {
    "LLM_BASE_URL": "LLM_BASE_URL",
    "LLM_API_KEY": "LLM_API_KEY",
    "LLM_BASE_MODEL": "LLM_BASE_MODEL",
    "LLM_LARGER_MODEL": "LLM_LARGER_MODEL",
}

def load_kaggle_secrets(secret_mapping, environ=None, client_factory=None):
    env = os.environ if environ is None else environ
    client = None
    diagnostics = {}
    for environment_name, secret_name in secret_mapping.items():
        if env.get(environment_name):
            diagnostics[environment_name] = "configured"
            continue
        if client is None:
            try:
                if client_factory is None:
                    from kaggle_secrets import UserSecretsClient
                    client_factory = UserSecretsClient
                client = client_factory()
            except Exception:
                client = False
        try:
            value = client.get_secret(secret_name) if client else None
        except Exception:
            value = None
        if value:
            env[environment_name] = value
            diagnostics[environment_name] = "configured"
        else:
            diagnostics[environment_name] = "missing"
    return diagnostics

secret_status = load_kaggle_secrets(SECRET_MAPPING)
print(secret_status)

## 8. Locate benchmark/index files in /kaggle/input

Explicit paths must be real files under `/kaggle/input`. When a value is unset, a unique conventional filename is discovered; ambiguous matches are blockers. No placeholders are created.

In [ ]:
INPUT_CANDIDATES = {
    "benchmark": ("qa_final.jsonl", "benchmark.jsonl"),
    "corpus": ("documents.jsonl", "corpus.jsonl"),
    "faiss_index": ("index.faiss",),
    "faiss_payloads": ("payloads.jsonl",),
    "faiss_manifest": ("manifest.json", "index_manifest.json"),
    "graph": ("graph.pkl", "graph.json", "knowledge_graph.pkl", "knowledge_graph.gpickle"),
}

def locate_input_file(value, label, *, input_root=KAGGLE_INPUT_ROOT):
    if value is not None:
        path = Path(value).expanduser().resolve()
        root = Path(input_root).resolve()
        if root != path and root not in path.parents:
            raise ValueError(f"{label} must be a file below /kaggle/input.")
        if not path.is_file():
            raise FileNotFoundError(f"{label} file is missing: {path.name}")
        return path
    root = Path(input_root)
    if not root.is_dir():
        return None
    matches = sorted({
        match.resolve()
        for filename in INPUT_CANDIDATES[label]
        for match in root.rglob(filename)
        if match.is_file()
    })
    if len(matches) > 1:
        raise RuntimeError(f"Multiple {label} files found; set its SOURCE value explicitly.")
    return matches[0] if matches else None

def locate_input_directory(value, label, *, input_root=KAGGLE_INPUT_ROOT):
    if value is None:
        return None
    path = Path(value).expanduser().resolve()
    root = Path(input_root).resolve()
    if root != path and root not in path.parents:
        raise ValueError(f"{label} must be a directory below /kaggle/input.")
    if not path.is_dir():
        raise FileNotFoundError(f"{label} directory is missing: {path.name}")
    return path

def validate_bm25_shard_root(value, *, expected_shards=10, input_root=KAGGLE_INPUT_ROOT):
    root = locate_input_directory(value, "bm25_shards", input_root=input_root)
    shard_dirs = sorted(path for path in root.glob("shard_*") if path.is_dir())
    incomplete = [
        shard.name for shard in shard_dirs
        if not (shard / "bm25_index.pkl").is_file()
        or not (shard / "bm25_metadata.pkl").is_file()
    ]
    if incomplete:
        raise FileNotFoundError("Incomplete BM25 shards: " + ", ".join(incomplete))
    if len(shard_dirs) != expected_shards:
        raise RuntimeError(
            f"Expected {expected_shards} BM25 shards, found {len(shard_dirs)} at {root}."
        )
    print({"bm25_mode": "independent_shards", "shard_count": len(shard_dirs), "root": str(root)})
    return root

faiss_index_path = locate_input_file(FAISS_INDEX_SOURCE, "faiss_index")
faiss_payloads_path = locate_input_file(FAISS_PAYLOADS_SOURCE, "faiss_payloads")
if FAISS_MANIFEST_SOURCE is not None:
    faiss_manifest_path = locate_input_file(FAISS_MANIFEST_SOURCE, "faiss_manifest")
elif faiss_index_path is not None:
    nearby_manifests = [faiss_index_path.parent / name for name in INPUT_CANDIDATES["faiss_manifest"]]
    faiss_manifest_path = next((path for path in nearby_manifests if path.is_file()), None)
else:
    faiss_manifest_path = None
bm25_shards_path = validate_bm25_shard_root(
    BM25_SHARDS_SOURCE, expected_shards=BM25_EXPECTED_SHARDS
)

input_sources = {
    "benchmark": locate_input_file(BENCHMARK_SOURCE, "benchmark"),
    "corpus": locate_input_file(CORPUS_SOURCE, "corpus"),
    "faiss_index": faiss_index_path,
    "faiss_payloads": faiss_payloads_path,
    "faiss_manifest": faiss_manifest_path,
    "bm25_shards": bm25_shards_path,
    "graph": locate_input_file(GRAPH_SOURCE, "graph"),
}
print({key: (str(value) if value else "missing") for key, value in input_sources.items()})

## 9. Prepare runtime path overrides

FAISS files may come from separate read-only datasets. This creates only links (or a copy fallback) in `/kaggle/working`, using the canonical filenames expected by production retrieval.

In [ ]:
def prepare_faiss_runtime_sources(index_source, payloads_source, manifest_source=None, *, working_root=KAGGLE_WORKING_ROOT):
    if index_source is None and payloads_source is None and manifest_source is None:
        return {}
    if index_source is None or payloads_source is None:
        raise RuntimeError("FAISS execution requires both index and payload files.")
    runtime_dir = Path(working_root) / "e2e_runtime_inputs" / "faiss"
    runtime_dir.mkdir(parents=True, exist_ok=True)
    mapping = {
        "faiss_index_source": (Path(index_source), runtime_dir / "index.faiss"),
        "faiss_payloads_source": (Path(payloads_source), runtime_dir / "payloads.jsonl"),
    }
    if manifest_source is not None:
        mapping["faiss_manifest_source"] = (Path(manifest_source), runtime_dir / "manifest.json")
    prepared = {}
    for key, (source, target) in mapping.items():
        if target.exists() or target.is_symlink():
            if target.resolve() != source.resolve():
                raise RuntimeError(f"Runtime input target already points elsewhere: {target.name}")
        else:
            try:
                target.symlink_to(source.resolve())
            except OSError:
                shutil.copy2(source, target)
        prepared[key] = target
    return prepared

prepared_faiss = prepare_faiss_runtime_sources(
    input_sources["faiss_index"],
    input_sources["faiss_payloads"],
    input_sources["faiss_manifest"],
)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print({key: str(value) for key, value in prepared_faiss.items()})

## 10. Import production repository APIs

This is intentionally the first project import.

In [ ]:
from scripts.run_ablation_config import (
    apply_runtime_path_overrides,
    load_ablation_configs,
    resolve_ablation_config,
    resolve_runtime_config,
    run_ablation_config,
    validate_ablation_config,
)
from evaluation.artifacts import (
    grouped_metric_rows,
    latency_summary_rows,
    load_run_artifacts,
    normalize_context_rows,
    normalize_trace_rows,
    prediction_table_rows,
    run_summary_rows,
    validate_run_parameters,
)
from evaluation.export import export_run_artifacts
from service.qa_service import run_preflight

## 11. Resolve selected config

In [ ]:
configs = load_ablation_configs(REPO_DIR / "configs" / "ablation_configs.yaml")
source_config = resolve_ablation_config(configs, CONFIG_NAME)
validate_ablation_config(source_config, config_name=CONFIG_NAME)

cuda_available = False
if _module_available("torch"):
    try:
        import torch
        cuda_available = bool(torch.cuda.is_available())
    except Exception:
        cuda_available = False
selected_device = "cuda" if USE_GPU_WHEN_AVAILABLE and cuda_available else "cpu"

resolved_config = apply_runtime_path_overrides(
    source_config,
    benchmark_source=input_sources["benchmark"],
    corpus_source=input_sources["corpus"],
    bm25_shards_source=input_sources["bm25_shards"],
    graph_source=input_sources["graph"],
    runs_root=RUNS_ROOT,
    selected_device=selected_device,
    **prepared_faiss,
)
resolved_config.setdefault("metadata", {})["kaggle_input_identities"] = {
    key: str(value) for key, value in input_sources.items() if value is not None
}
print({"selected_config": CONFIG_NAME, "selected_device": selected_device, "config": resolved_config})

## 12. Run preflight

A structurally valid configuration is not called runtime-ready when packages, files, credentials, or provider settings are missing.

In [ ]:
import json
import os
import shutil
from pathlib import Path

service_preflight = run_preflight(
    resolved_config,
    config_name=CONFIG_NAME,
    project_root=REPO_DIR,
)
# Preflight returns a redacted copy for display. Never execute that copy:
# keys such as max_output_tokens may otherwise be mistaken for credentials.
execution_config = resolve_runtime_config(resolved_config)

def _writable(path):
    try:
        Path(path).mkdir(parents=True, exist_ok=True)
        probe = Path(path) / ".write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except OSError:
        return False

dense = execution_config["retrieval"]["dense"]
graph = execution_config["retrieval"].get("graph", {})
generation = execution_config["generation"]
required_packages_ready = not dependency_status["missing_after_setup"]
readiness = [
    {"check": "repository cloned", "status": "ready" if valid_checkout(REPO_DIR) else "blocked"},
    {"check": "Git commit resolved", "status": "ready" if repository_info.get("git_commit") else "blocked"},
    {"check": "Kaggle environment", "status": "ready" if IS_KAGGLE else "warning"},
    {"check": "GPU availability", "status": "ready" if cuda_available else "not required"},
    {"check": "selected device", "status": "ready", "detail": selected_device},
    {"check": "required package imports", "status": "ready" if required_packages_ready else "blocked"},
    {"check": "optional pandas display", "status": "ready" if _module_available("pandas") else "warning"},
    {"check": "config validity", "status": "ready"},
    {"check": "config execution/defer status", "status": "ready" if service_preflight.runnable else service_preflight.status.replace("runtime-", "")},
    {"check": "benchmark availability", "status": "ready" if input_sources["benchmark"] else "blocked"},
    {"check": "corpus availability", "status": "ready" if input_sources["corpus"] else "blocked"},
    {"check": "FAISS index availability", "status": "ready" if not dense.get("enabled") or dense.get("backend") != "faiss" or input_sources["faiss_index"] else "blocked"},
    {"check": "payload availability", "status": "ready" if not dense.get("enabled") or dense.get("backend") != "faiss" or input_sources["faiss_payloads"] else "blocked"},
    {"check": "graph artifacts when enabled", "status": "ready" if not graph.get("enabled") or input_sources["graph"] else ("not required" if not graph.get("enabled") else "blocked")},
    {"check": "model selector", "status": "ready" if not str(generation.get("model", "")).startswith("env:") else "blocked"},
    {"check": "provider URL", "status": "not required" if generation.get("provider") != "openai_compatible" else ("ready" if os.environ.get("LLM_BASE_URL") else "blocked")},
    {"check": "API credential", "status": "not required" if generation.get("provider") == "reference" else ("ready" if os.environ.get(str(generation.get("api_key_env") or "LLM_API_KEY")) else "blocked")},
    {"check": "runs root writability", "status": "ready" if _writable(RUNS_ROOT) else "blocked"},
    {"check": "export root writability", "status": "ready" if _writable(EXPORT_ROOT) else "blocked"},
    {"check": "selected run mode", "status": "ready", "detail": RUN_MODE},
]
print(json.dumps(readiness, indent=2))
for check in service_preflight.checks:
    print({"production_check": check.name, "status": check.status, "detail": check.message})

## 13. Execute inspect, smoke, or full mode

Smoke forwards exactly `SMOKE_LIMIT`; full never downgrades. Both use the named-config production runner.

In [ ]:
validate_run_parameters(
    RUN_MODE,
    smoke_limit=SMOKE_LIMIT,
    existing_run_dir=Path(EXISTING_RUN_DIR) if EXISTING_RUN_DIR else None,
)
artifacts = None
outcome = None
if RUN_MODE == "inspect":
    if EXISTING_RUN_DIR is None:
        print("Inspect mode: no existing run configured; no execution was performed.")
    else:
        inspect_dir = Path(EXISTING_RUN_DIR).resolve()
        allowed_roots = (Path("/kaggle/input").resolve(), Path("/kaggle/working").resolve())
        if not any(root == inspect_dir or root in inspect_dir.parents for root in allowed_roots):
            raise ValueError("EXISTING_RUN_DIR must be under /kaggle/input or /kaggle/working.")
        artifacts = load_run_artifacts(inspect_dir)
else:
    blockers = [row["check"] for row in readiness if row["status"] == "blocked"]
    if not service_preflight.runnable:
        blockers.extend(service_preflight.blockers)
    if blockers:
        raise RuntimeError("Run blocked by preflight: " + "; ".join(dict.fromkeys(blockers)))
    outcome = run_ablation_config(
        CONFIG_NAME,
        config_file=REPO_DIR / "configs" / "ablation_configs.yaml",
        output_root=RUNS_ROOT,
        limit=SMOKE_LIMIT if RUN_MODE == "smoke" else None,
        project_root=REPO_DIR,
        resolved_config_override=execution_config,
    )
    if outcome.status not in {"completed"}:
        raise RuntimeError(f"Production run ended with status {outcome.status}: {outcome.error or 'N/A'}")
    artifacts = load_run_artifacts(outcome.output_dir, require_completed=True)
print({"mode": RUN_MODE, "run_directory": str(artifacts.run_dir) if artifacts else None})

## 14. Export artifacts under /kaggle/working

Only canonical run outputs are copied. Benchmark/corpus/index/model-cache files and secrets are excluded, and existing exports are never overwritten.

In [ ]:
export_result = None
if outcome is not None and artifacts is not None:
    credential_env_names = set()
    for section_name in ("generation", "judge"):
        section = execution_config.get(section_name, {})
        provider = section.get("provider") if isinstance(section, dict) else None
        if provider in (None, "none", "reference"):
            continue
        default_env = "GEMINI_API_KEY" if provider == "gemini" else "LLM_API_KEY"
        credential_env_names.add(str(section.get("api_key_env") or default_env))
    sensitive_values = tuple(
        os.environ.get(name, "") for name in sorted(credential_env_names)
    )
    export_result = export_run_artifacts(
        artifacts.run_dir,
        EXPORT_ROOT,
        create_zip=CREATE_EXPORT_ZIP,
        sensitive_values=sensitive_values,
    )
    print({
        "export_directory": str(export_result.directory),
        "export_zip": str(export_result.archive) if export_result.archive else None,
        "files": list(export_result.files),
    })
else:
    print("No new run was produced; export was skipped.")

## 15. Display results and reproducibility data

Pandas is optional and imported only here. If it is incompatible with NumPy/pyarrow, execution and export remain complete and Python rows are shown instead. Missing values stay `None`/`N/A`; failures and skips remain visible. Context references are evidence pointers, not automatically verified formal citations. Hidden reasoning and raw stack traces are never displayed.

In [ ]:
def display_rows(title, rows):
    print(f"\n## {title}")
    values = list(rows)
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(values))
    except Exception as exc:
        print(f"Optional pandas display unavailable ({type(exc).__name__}). Using Python rows; pyarrow is not required.")
        for row in values:
            print(row)

if artifacts is None:
    print("No artifacts selected. Set EXISTING_RUN_DIR or choose smoke/full after preflight.")
else:
    predictions = list(artifacts.predictions)
    metrics = artifacts.metrics or {}
    selected = predictions[0] if predictions else None
    agent_mode = (artifacts.resolved_config or {}).get("agent", {}).get("mode")

    display_rows("safe resolved config", [{"config": artifacts.resolved_config}])
    display_rows("run summary", run_summary_rows(artifacts, repo_root=REPO_DIR))
    display_rows("per-case results", prediction_table_rows(predictions, agent_mode=agent_mode))
    display_rows("selected-case details", [selected] if selected else [])
    contexts = normalize_context_rows((selected or {}).get("retrieved_context") or [])
    display_rows("retrieved contexts", contexts)
    display_rows("rank and scores", [{key: row.get(key) for key in ("rank", "score", "vector_score", "reranker_score")} for row in contexts])
    display_rows("retrieved evidence/context references", [{key: row.get(key) for key in ("rank", "document_id", "chunk_id", "context_reference", "source_path")} for row in contexts])
    display_rows("overall E2E metrics", [metrics.get("overall") or {}])
    display_rows("retrieval metrics", [artifacts.retrieval_metrics or metrics.get("retrieval") or {}])
    display_rows("agent metrics", [metrics.get("agent") or {}])
    display_rows("category metrics", grouped_metric_rows(metrics, "by_category"))
    display_rows("answer-type metrics", grouped_metric_rows(metrics, "by_answer_type"))
    display_rows("difficulty metrics", grouped_metric_rows(metrics, "by_difficulty"))
    display_rows("stage-level latency", latency_summary_rows(artifacts.latency))
    display_rows("agent trace", normalize_trace_rows((selected or {}).get("agent_trace") or []))
    display_rows("failure analysis", list(artifacts.errors))
    display_rows("denominator analysis", [metrics.get("counts") or {}])
    display_rows("reproducibility metadata", [{
        "git_commit": artifacts.manifest.get("git_commit"),
        "config_hash": artifacts.manifest.get("config_hash"),
        "benchmark_identity": artifacts.manifest.get("benchmark_path"),
        "corpus_identity": artifacts.manifest.get("corpus_path"),
        "index_identity": artifacts.manifest.get("index_path"),
        "selected_device": (artifacts.resolved_config or {}).get("runtime", {}).get("selected_device"),
    }])
    display_rows("artifact paths", [{"path": str(artifacts.run_dir / name)} for name in sorted(path.name for path in artifacts.run_dir.iterdir() if path.is_file())])
    for diagnostic in artifacts.diagnostics:
        print({"artifact_diagnostic": diagnostic})